Código para preprocesar el texto de El Quijote, carácter a carácter, como alternativa a usar `TextVectorization` directamente en la red:



In [7]:
import tensorflow as tf
import numpy as np

# 1. Cargar el texto
filepath = "../datos/el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Crear vocabulario
vocab = sorted(set(el_quijote_text))
char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = {index: char for index, char in enumerate(vocab)}

# 3. Convertir el texto a índices numéricos
text_as_int = np.array([char_to_index[char] for char in el_quijote_text])

# 4. Crear secuencias de entrenamiento
seq_length = 100  # Longitud de las secuencias
examples_per_epoch = len(el_quijote_text) // (seq_length + 1)

# Crear dataset de tf.data
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

# Función para dividir la secuencia en entrada y objetivo
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# 5. Preparar el dataset para el entrenamiento
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# Ahora 'dataset' está listo para ser usado en el entrenamiento de la red.
# Puedes usar 'len(vocab)' para el tamaño del vocabulario en la capa Embedding.



**Explicación del código:**

1.  **Cargar el texto:** Lee el archivo "el\_quijote.txt" y guarda el contenido en la variable `el_quijote_text`.
2.  **Crear vocabulario:** Crea un conjunto ordenado de caracteres únicos en el texto. Luego, crea diccionarios para mapear caracteres a índices y viceversa.
3.  **Convertir el texto a índices numéricos:** Convierte cada carácter del texto a su índice correspondiente en el vocabulario. El resultado es un array NumPy llamado `text_as_int`.
4.  **Crear secuencias de entrenamiento:**
    *   Define la longitud de las secuencias (`seq_length`).
    *   Calcula cuántas secuencias completas se pueden crear a partir del texto.
    *   Crea un `tf.data.Dataset` a partir del array de índices.
    *   Divide el dataset en secuencias de longitud `seq_length + 1`.
    *   Define una función (`split_input_target`) para dividir cada secuencia en texto de entrada (todos los caracteres excepto el último) y texto objetivo (todos los caracteres excepto el primero).
    *   Aplica la función `split_input_target` al dataset.
5.  **Preparar el dataset para el entrenamiento:**
    *   Define el tamaño del lote (`BATCH_SIZE`) y el tamaño del buffer para el shuffle (`BUFFER_SIZE`).
    *   Shufflea el dataset.
    *   Divide el dataset en lotes.
    *   Prefetch para mejorar el rendimiento.

**Cómo usar este código:**

1.  Guarda este código en un archivo Python (por ejemplo, `preprocess.py`).
2.  Asegúrate de tener el archivo "el\_quijote.txt" en el mismo directorio.
3.  Ejecuta el script.
4.  Ahora puedes usar la variable `dataset` para entrenar tu red neuronal. También necesitarás `len(vocab)` para definir el tamaño del vocabulario en la capa `Embedding` de tu modelo.

**Ventajas de este enfoque:**

*   **Control total:** Tienes control total sobre el proceso de preprocesamiento.
*   **Flexibilidad:** Puedes modificar fácilmente el código para cambiar la longitud de las secuencias, el tamaño del lote, etc.
*   **Depuración:** Es más fácil depurar el proceso de preprocesamiento.

**Desventajas:**

*   **Más código:** Requiere escribir más código que usar `TextVectorization` directamente.
*   **Menos integración:** No está tan integrado con Keras como `TextVectorization`.

En general, este enfoque es una buena alternativa si necesitas más control sobre el proceso de preprocesamiento o si quieres realizar transformaciones más complejas en el texto.

In [8]:
import tensorflow as tf

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(batch_shape=(batch_size, None)),  # Añadido
        tf.keras.layers.Embedding(vocab_size, embedding_dim),
        tf.keras.layers.GRU(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

# Hiperparámetros
VOCAB_SIZE = len(vocab)  # Definido en el código de preprocesamiento
EMBEDDING_DIM = 256
RNN_UNITS = 1024

# Reconstruir el modelo
model = build_model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_units=RNN_UNITS,
    batch_size=BATCH_SIZE)

# Función de pérdida
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])



model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (64, None, 256)        │        22,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (64, None, 1024)       │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (64, None, 89)         │        91,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,052,313 (15.46 MB)

 Trainable params: 4,052,313 (15.46 MB)

 Non-trainable params: 0 (0.00 B)



**Explicación:**

1.  **`build_model(vocab_size, embedding_dim, rnn_units, batch_size)`:**
    *   Esta función define la arquitectura del modelo.
    *   **`tf.keras.layers.Embedding`:** La capa de embedding toma el tamaño del vocabulario (`vocab_size`) y la dimensión del embedding (`embedding_dim`) como argumentos.  `batch_input_shape` especifica el tamaño del lote y la longitud de la secuencia de entrada.  `None` indica que la longitud de la secuencia puede variar.
    *   **`tf.keras.layers.GRU`:** La capa GRU (Gated Recurrent Unit) es una capa recurrente que procesa la secuencia de entrada.  `return_sequences=True` indica que la capa debe devolver la secuencia completa de salidas, no solo la última.  `stateful=True` es crucial para mantener el estado entre lotes.  `recurrent_initializer='glorot_uniform'` es una buena práctica para inicializar los pesos recurrentes.
    *   **`tf.keras.layers.Dense`:** La capa densa es una capa completamente conectada que produce la salida del modelo. El número de unidades es igual al tamaño del vocabulario (`vocab_size`), y la salida representa la probabilidad de cada carácter en el vocabulario.
2.  **Hiperparámetros:**
    *   `VOCAB_SIZE`: El tamaño del vocabulario (número de caracteres únicos).  **Importante:** Asegúrate de que `vocab` esté definido correctamente en tu código de preprocesamiento.
    *   `EMBEDDING_DIM`: La dimensión del espacio de embedding.  Un valor común es 256.
    *   `RNN_UNITS`: El número de unidades en la capa GRU.  Un valor común es 1024.
3.  **Construcción del modelo:**
    *   Se llama a la función `build_model` para crear el modelo, pasando los hiperparámetros definidos.
4.  **Función de pérdida:**
    *   La función `loss` calcula la pérdida entre las predicciones del modelo y las etiquetas reales.  `sparse_categorical_crossentropy` es adecuada para problemas de clasificación multiclase donde las etiquetas son enteros.  `from_logits=True` indica que la salida del modelo no está normalizada (es decir, no es una distribución de probabilidad).
5.  **Compilación del modelo:**
    *   Se compila el modelo, especificando el optimizador (`adam`) y la función de pérdida (`loss`).
6.  **Resumen del modelo:**
    *   `model.summary()` imprime un resumen de la arquitectura del modelo, incluyendo el número de parámetros entrenables.

**Puntos importantes:**

*   **`stateful=True`:** Este es un parámetro clave para el entrenamiento de modelos generativos de texto.  Indica que la capa GRU debe mantener su estado interno entre lotes.  Esto permite que el modelo aprenda dependencias a largo plazo en el texto.
*   **`batch_input_shape`:**  Especifica el tamaño del lote de entrada.  Esto es necesario para que la capa de embedding sepa cómo inicializar sus pesos.
*   **`VOCAB_SIZE`:** Asegúrate de que `VOCAB_SIZE` esté definido correctamente y que coincida con el tamaño del vocabulario que creaste en el código de preprocesamiento.
*   **Ajuste de hiperparámetros:**  Los valores de `EMBEDDING_DIM` y `RNN_UNITS` son solo ejemplos.  Puedes experimentar con diferentes valores para ver cómo afectan el rendimiento del modelo.

**Cómo usar este código:**

1.  Asegúrate de haber ejecutado el código de preprocesamiento que te proporcioné anteriormente.
2.  Copia este código en tu script de entrenamiento.
3.  Ejecuta el script.

Ahora tendrás un modelo Keras listo para ser entrenado con los datos preprocesados.  Recuerda que debes pasar el dataset `dataset` (creado en el código de preprocesamiento) a la función `model.fit()`.




In [9]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=0.00001
)

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "my_quijote_model", monitor="val_accuracy", save_best_only=True
)
history = model.fit(
    train_set,
    validation_data=valid_set,
    epochs=10,
    callbacks=[model_ckpt, early_stopping, reduce_lr],
)

ValueError: The filepath provided must end in `.keras` (Keras model format). Received: filepath=my_quijote_model

El código anterior no funcionará directamente con el modelo definido para entrenarlo. Hay un problema de compatibilidad entre el formato de los datos generados por el código de preprocesamiento y la forma en que el modelo espera recibir las entradas.

El código de preprocesamiento genera un `tf.data.Dataset` donde cada elemento es un par `(input_text, target_text)`, donde `input_text` y `target_text` son tensores que representan secuencias de índices de caracteres. Además, el modelo se ha definido como `stateful`, lo que implica que el tamaño del lote debe ser fijo y conocido de antemano.

Sin embargo, el código de entrenamiento que se usó anteriormente (con `TextVectorization`) no era `stateful` y no tenía un tamaño de lote fijo.

Para que el entrenamiento funcione correctamente con el modelo `stateful` y los datos preprocesados, es necesario realizar los siguientes ajustes:

1.  **Resetear los estados del modelo:** Dado que el modelo es `stateful`, es necesario resetear sus estados al inicio de cada época. Esto se puede hacer definiendo un callback personalizado.
2.  **Asegurarse de que el tamaño del lote sea consistente:** El tamaño del lote definido en el código de preprocesamiento (`BATCH_SIZE`) debe coincidir con el tamaño del lote especificado al construir el modelo.
3.  **Entrenar el modelo con el dataset preprocesado:** Usar el `dataset` creado con el código de preprocesamiento en la función `model.fit()`.

Este es el código de entrenamiento modificado para que sea compatible con el modelo `stateful` y los datos preprocesados:



In [ ]:
import tensorflow as tf

# Callback para resetear los estados al inicio de cada época
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

# Crear el callback
reset_states_callback = ResetStatesCallback()

# Entrenar el modelo
EPOCHS = 50  # Ajusta el número de épocas según sea necesario

history = model.fit(dataset, epochs=EPOCHS, callbacks=[reset_states_callback])



**Explicación de los cambios:**

*   **`ResetStatesCallback`:** Este callback se encarga de resetear los estados del modelo al inicio de cada época. Esto es esencial para los modelos `stateful`, ya que de lo contrario, el modelo "recordaría" información de los lotes anteriores, lo que podría afectar negativamente el entrenamiento.
*   **`model.fit(dataset, ...)`:** La función `model.fit()` ahora recibe el `dataset` creado con el código de preprocesamiento.

Con estos cambios, el código de entrenamiento debería funcionar correctamente con el modelo `stateful` y los datos preprocesados. Recuerda ajustar el número de épocas (`EPOCHS`) según sea necesario.

Se puede reutilizar los callbacks de `EarlyStopping`, `ReduceLROnPlateau` y `ModelCheckpoint` con el modelo `stateful`, pero es importante tener en cuenta algunas consideraciones:

*   **`EarlyStopping`:** Este callback funciona sin problemas con modelos `stateful`. Simplemente monitorea una métrica (por ejemplo, la pérdida de validación) y detiene el entrenamiento si la métrica deja de mejorar durante un cierto número de épocas.
*   **`ReduceLROnPlateau`:** Este callback también funciona bien con modelos `stateful`. Reduce la tasa de aprendizaje cuando una métrica deja de mejorar, lo que puede ayudar al modelo a converger más rápido.
*   **`ModelCheckpoint`:** Este callback guarda el modelo en un archivo después de cada época (o solo cuando mejora una métrica). Con modelos `stateful`, es importante guardar los pesos del modelo, no el modelo completo, ya que guardar el modelo completo puede causar problemas al cargarlo posteriormente.

Aquí vemos cómo puedes usar estos callbacks con el modelo `stateful`:



In [ ]:
import tensorflow as tf

# ... (Código de preprocesamiento, definición del modelo y ResetStatesCallback como se proporcionó anteriormente) ...

# Callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss',  # O 'val_loss' si tienes datos de validación
    patience=10,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss',  # O 'val_loss'
    factor=0.2,
    patience=5,
    min_lr=0.0001
)

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='checkpoints/quijote2_model_weights',  # Guarda solo los pesos
    save_weights_only=True,
    monitor='loss',  # O 'val_loss'
    save_best_only=True
)

# Entrenar el modelo
EPOCHS = 50  # Ajusta el número de épocas según sea necesario

history = model.fit(dataset, epochs=EPOCHS,
                    callbacks=[reset_states_callback, early_stopping, reduce_lr, model_checkpoint])



**Puntos importantes:**

*   **`save_weights_only=True` en `ModelCheckpoint`:** Es crucial guardar solo los pesos del modelo, no el modelo completo. Esto evita problemas al cargar el modelo posteriormente.
*   **`filepath` en `ModelCheckpoint`:** Especifica una ruta donde se guardarán los pesos del modelo. Asegúrate de crear el directorio "checkpoints" antes de ejecutar el código.
*   **`monitor` en los callbacks:** Ajusta la métrica que se monitorea en los callbacks (`loss` o `val_loss`) según si estás usando datos de validación o no.
*   **`reset_states_callback`:** Este callback es esencial para modelos `stateful` y debe incluirse siempre.

Con estos callbacks, puedes entrenar tu modelo `stateful` de manera más eficiente y evitar el sobreajuste.

In [ ]:
# Reconstruir el modelo con batch_size=BATCH_SIZE (igual que en el entrenamiento)
model = build_model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_units=RNN_UNITS,
    batch_size=BATCH_SIZE
)

# Cargar los pesos guardados
model.load_weights('checkpoints/quijote2_model_weights')

### Con Pytorch

Aquí te muestro cómo realizar el preprocesamiento, la creación del modelo y el entrenamiento con PyTorch, adaptando el enfoque de modelo `stateful` carácter a carácter:



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# 1. Cargar el texto
filepath = "./el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Crear vocabulario
vocab = sorted(set(el_quijote_text))
char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = {index: char for index, char in enumerate(vocab)}

# 3. Convertir el texto a índices numéricos
text_as_int = np.array([char_to_index[char] for char in el_quijote_text])

# 4. Crear secuencias de entrenamiento
seq_length = 100

# 5. Dataset y DataLoader
class QuijoteDataset(Dataset):
    def __init__(self, text_as_int, seq_length):
        self.text_as_int = text_as_int
        self.seq_length = seq_length
        self.examples_per_epoch = len(text_as_int) // (seq_length + 1)
        self.num_sequences = self.examples_per_epoch

        # Generar los datos
        self.input_seqs = []
        self.target_seqs = []
        for i in range(self.num_sequences):
            start_index = i * (seq_length + 1)
            end_index = start_index + seq_length + 1
            chunk = text_as_int[start_index:end_index]
            input_text = chunk[:-1]
            target_text = chunk[1:]
            self.input_seqs.append(input_text)
            self.target_seqs.append(target_text)

    def __len__(self):
        return len(self.input_seqs)

    def __getitem__(self, idx):
        return torch.tensor(self.input_seqs[idx], dtype=torch.long), torch.tensor(self.target_seqs[idx], dtype=torch.long)

BATCH_SIZE = 64
dataset = QuijoteDataset(text_as_int, seq_length)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, drop_last=True)

# 6. Definir el modelo
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super(CharRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.dense = nn.Linear(rnn_units, vocab_size)

        self.rnn_units = rnn_units

    def forward(self, x, hidden):
        # x shape: (batch_size, seq_length)
        # hidden shape: (1, batch_size, rnn_units)

        embedded = self.embedding(x)  # shape: (batch_size, seq_length, embedding_dim)
        output, hidden = self.gru(embedded, hidden)  # output shape: (batch_size, seq_length, rnn_units)
        output = self.dense(output)  # shape: (batch_size, seq_length, vocab_size)
        return output, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.rnn_units)

# 7. Hiperparámetros
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 256
RNN_UNITS = 1024
LEARNING_RATE = 0.001
NUM_EPOCHS = 50

# 8. Instanciar el modelo
model = CharRNN(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS)

# 9. Definir optimizador y función de pérdida
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# 10. Dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 11. Loop de entrenamiento
for epoch in range(NUM_EPOCHS):
    # Inicializar el estado oculto
    hidden = model.init_hidden(BATCH_SIZE).to(device)

    for batch_idx, (input_data, target_data) in enumerate(dataloader):
        input_data = input_data.to(device)
        target_data = target_data.to(device)

        # Limpiar los gradientes
        optimizer.zero_grad()

        # Forward pass
        output, hidden = model(input_data, hidden)

        # Calcular la pérdida
        loss = criterion(output.view(-1, VOCAB_SIZE), target_data.view(-1))

        # Backward pass y optimización
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}')

    # Detach hidden state (important for stateful RNNs)
    hidden = hidden.detach()



**Explicación detallada:**

1.  **Importar librerías:** Importa las librerías necesarias de PyTorch.
2.  **Cargar el texto:**  Igual que en el ejemplo de TensorFlow, carga el texto de El Quijote desde el archivo.
3.  **Crear vocabulario:** Crea el vocabulario de caracteres únicos.
4.  **Convertir el texto a índices numéricos:** Convierte el texto a una representación numérica usando el vocabulario.
5.  **Dataset y DataLoader:**
    *   **`QuijoteDataset`:**  Define una clase `Dataset` personalizada para manejar los datos.  Esta clase se encarga de:
        *   Recibir el texto convertido a índices (`text_as_int`) y la longitud de la secuencia (`seq_length`).
        *   Dividir el texto en secuencias de entrada y salida.
        *   Implementar los métodos `__len__` (para obtener la longitud del dataset) y `__getitem__` (para obtener un elemento del dataset).
    *   **`DataLoader`:**  Crea un `DataLoader` para iterar sobre el dataset en lotes.  `drop_last=True` es importante para asegurar que todos los lotes tengan el mismo tamaño, lo cual es necesario para un modelo `stateful`.
6.  **Definir el modelo (`CharRNN`):**
    *   **`__init__`:**  Define las capas del modelo:
        *   `nn.Embedding`: Capa de embedding para convertir los índices de caracteres en vectores densos.
        *   `nn.GRU`: Capa GRU (Gated Recurrent Unit) para procesar las secuencias.  `batch_first=True` indica que la entrada tendrá la forma `(batch_size, seq_length, embedding_dim)`.
        *   `nn.Linear`: Capa lineal (fully connected) para mapear la salida de la GRU al tamaño del vocabulario.
    *   **`forward`:**  Define el paso forward del modelo:
        *   Aplica la capa de embedding a la entrada (`x`).
        *   Pasa la salida de la capa de embedding y el estado oculto (`hidden`) a la capa GRU.
        *   Aplica la capa lineal a la salida de la GRU.
        *   Devuelve la salida y el nuevo estado oculto.
    *   **`init_hidden`:**  Inicializa el estado oculto de la GRU con ceros.  Este método se llama al inicio de cada época.
7.  **Hiperparámetros:** Define los hiperparámetros del modelo y del entrenamiento.
8.  **Instanciar el modelo:** Crea una instancia del modelo `CharRNN`.
9.  **Definir optimizador y función de pérdida:**
    *   **`optim.Adam`:**  Crea un optimizador Adam para actualizar los pesos del modelo.
    *   **`nn.CrossEntropyLoss`:**  Crea una función de pérdida CrossEntropyLoss, que es adecuada para problemas de clasificación multiclase.
10. **Dispositivo (GPU si está disponible):** Mueve el modelo al dispositivo (GPU si está disponible, de lo contrario, CPU).
11. **Loop de entrenamiento:**
    *   Itera sobre las épocas.
    *   **Inicializar el estado oculto:** Llama a `model.init_hidden` al inicio de cada época para inicializar el estado oculto de la GRU.  Esto es crucial para un modelo `stateful`.
    *   Itera sobre los lotes del `DataLoader`.
    *   Mueve los datos al dispositivo.
    *   **Limpiar los gradientes:** Llama a `optimizer.zero_grad()` para limpiar los gradientes acumulados de la iteración anterior.
    *   **Forward pass:** Pasa los datos y el estado oculto al modelo para obtener la salida y el nuevo estado oculto.
    *   **Calcular la pérdida:** Calcula la pérdida entre la salida del modelo y las etiquetas reales.  `output.view(-1, VOCAB_SIZE)` y `target_data.view(-1)` aplanan la salida y las etiquetas para que sean compatibles con `CrossEntropyLoss`.
    *   **Backward pass y optimización:** Llama a `loss.backward()` para calcular los gradientes y a `optimizer.step()` para actualizar los pesos del modelo.
    *   Imprime la pérdida al final de cada época.
    *   **`hidden = hidden.detach()`:**  **¡MUY IMPORTANTE!**  Desconecta el estado oculto del grafo computacional después de cada lote.  Esto evita que los gradientes se propaguen a través de todos los lotes de la época, lo cual consumiría mucha memoria y haría que el entrenamiento fuera muy lento.  Al hacer `hidden.detach()`, se crea una copia del estado oculto que no está conectada al grafo computacional, por lo que los gradientes no se propagarán a través de ella.

**Puntos clave para un modelo `stateful` en PyTorch:**

*   **`drop_last=True` en `DataLoader`:**  Asegura que todos los lotes tengan el mismo tamaño.
*   **Inicializar el estado oculto al inicio de cada época:**  Llama a `model.init_hidden()` al inicio de cada época para resetear el estado oculto.
*   **`hidden = hidden.detach()`:**  Desconecta el estado oculto del grafo computacional después de cada lote para evitar el consumo excesivo de memoria.
*   **Tamaño del lote consistente:** El tamaño del lote debe ser el mismo durante todo el entrenamiento.

Este código proporciona una base sólida para entrenar un modelo generativo de texto `stateful` con PyTorch. Puedes experimentar con diferentes hiperparámetros, arquitecturas de modelo y técnicas de regularización para mejorar el rendimiento.

## Con Fast.ai

Ok, aquí te muestro cómo adaptar el enfoque de generación de texto carácter a carácter con `fast.ai`.  `fast.ai` simplifica mucho el proceso de entrenamiento de redes neuronales, pero requiere una estructura de datos específica.



In [ ]:
from fastai.text.all import *
import numpy as np

# 1. Cargar el texto
filepath = "./el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Crear vocabulario
vocab = sorted(set(el_quijote_text))
char_to_index = {char: index for index, char in enumerate(vocab)}
index_to_char = {index: char for index, char in enumerate(vocab)}

# 3. Convertir el texto a índices numéricos
text_as_int = [char_to_index[char] for char in el_quijote_text]

# 4. Preparar los datos para fastai
# fastai necesita un DataLoader con un formato específico.
# Vamos a crear un DataLoaders directamente a partir del texto.

# Definir la longitud de la secuencia
seq_length = 100

# Crear un DataLoaders
bs=64 #Tamaño del batch
sl=seq_length #Longitud de la secuencia
dls = DataLoaders.from_label_func(
    '.',
    is_lm=True,
    valid_pct=0.2,  # Porcentaje para validación
    label_func=lambda x: x,
    text_vocab=vocab,
    bs=bs,
    seq_len=sl
)

# Crear un modelo
learn = language_model_learner(
    dls, AWD_LSTM, drop_mult=0.3,
    metrics=[accuracy, Perplexity()],
    pretrained=False) #No usar modelo preentrenado

# Encontrar una buena tasa de aprendizaje
learn.lr_find()

# Entrenar el modelo
learn.fit_one_cycle(1, 1e-2) #Ajustar la tasa de aprendizaje

# Generar texto
TEXT = "En un lugar de la Mancha"
N_WORDS = 40
N_SENTENCES = 2

print("\n".join(learn.predict(TEXT, N_WORDS, temperature=0.75) for _ in range(N_SENTENCES)))



**Explicación detallada:**

1.  **Importar librerías:** Importa las librerías necesarias de `fastai`.
2.  **Cargar el texto:** Carga el texto de El Quijote desde el archivo.
3.  **Crear vocabulario:** Crea el vocabulario de caracteres únicos.
4.  **Convertir el texto a índices numéricos:** Convierte el texto a una representación numérica usando el vocabulario.
5.  **Preparar los datos para fastai:**
    *   **`DataLoaders.from_label_func`:** Esta función crea un `DataLoaders` a partir de una función que asigna etiquetas a los datos. En este caso, estamos usando `is_lm=True` para indicar que se trata de un modelo de lenguaje, lo que significa que la etiqueta es el mismo texto desplazado un carácter.
        *   `'.'` : Especifica el path de los datos. En este caso, como le pasamos el texto directamente, le pasamos el directorio actual.
        *   `is_lm=True`: Especifica que es un modelo de lenguaje.
        *   `valid_pct=0.2`: Especifica el porcentaje de datos para validación.
        *   `label_func=lambda x: x`: La función de etiquetado es la identidad, ya que el modelo de lenguaje predice el siguiente token.
        *   `text_vocab=vocab`: El vocabulario que hemos creado.
        *   `bs=bs`: El tamaño del batch.
        *   `seq_len=sl`: La longitud de la secuencia.
6.  **Crear un modelo:**
    *   **`language_model_learner`:** Esta función crea un modelo de lenguaje usando la arquitectura AWD\_LSTM (una variante de LSTM).
        *   `dls`: Los DataLoaders que hemos creado.
        *   `AWD_LSTM`: La arquitectura del modelo.
        *   `drop_mult=0.3`: Multiplicador para la cantidad de dropout.
        *   `metrics=[accuracy, Perplexity()]`: Las métricas que queremos monitorizar.
        *   `pretrained=False`: No usar un modelo preentrenado.
7.  **Encontrar una buena tasa de aprendizaje:**
    *   **`learn.lr_find()`:** Esta función ayuda a encontrar una buena tasa de aprendizaje para el modelo.
8.  **Entrenar el modelo:**
    *   **`learn.fit_one_cycle`:** Esta función entrena el modelo usando la técnica "one cycle policy", que ajusta la tasa de aprendizaje de forma dinámica durante el entrenamiento.
        *   `1`: El número de ciclos de entrenamiento.
        *   `1e-2`: La tasa de aprendizaje máxima.
9.  **Generar texto:**
    *   **`learn.predict`:** Esta función genera texto a partir de un prompt.
        *   `TEXT`: El prompt inicial.
        *   `N_WORDS`: El número de palabras a generar.
        *   `temperature=0.75`: Controla la aleatoriedad de la generación. Valores más bajos hacen que el texto sea más predecible, mientras que valores más altos lo hacen más aleatorio.
    *   `print("\n".join(learn.predict(TEXT, N_WORDS, temperature=0.75) for _ in range(N_SENTENCES)))`: Imprime el texto generado.

**Puntos clave:**

*   **`DataLoaders`:** `fastai` usa `DataLoaders` para manejar los datos. Es importante crear el `DataLoaders` correctamente para que el modelo pueda entrenar.
*   **`language_model_learner`:** Esta función simplifica la creación de modelos de lenguaje.
*   **`fit_one_cycle`:** Esta función entrena el modelo usando una política de tasa de aprendizaje que suele dar buenos resultados.
*   **`predict`:** Esta función genera texto a partir de un prompt.

Este código proporciona una forma sencilla de entrenar un modelo generativo de texto carácter a carácter con `fastai`. Puedes experimentar con diferentes hiperparámetros, arquitecturas de modelo y técnicas de regularización para mejorar el rendimiento.

## Generación palabra a palabra

#### Keras

Aquí tienes el código para la generación de texto palabra a palabra con Keras, basado en el código anterior de generación carácter a carácter:



In [ ]:
import tensorflow as tf
import numpy as np

# 1. Cargar el texto
filepath = "./el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Tokenización por palabras
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=None, oov_token="<unk>")
tokenizer.fit_on_texts([el_quijote_text])
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1

# 3. Convertir texto a secuencias de índices
sequences = tokenizer.texts_to_sequences([el_quijote_text])[0]

# 4. Crear secuencias de entrenamiento
seq_length = 100
examples_per_epoch = len(sequences) // (seq_length + 1)

# 5. Crear dataset de tf.data
data = tf.data.Dataset.from_tensor_slices(sequences)

# Función para crear secuencias
def create_sequences(data):
    seqs = data.batch(seq_length + 1, drop_remainder=True)

    def split_input_target(chunk):
        input_text = chunk[:-1]
        target_text = chunk[1:]
        return input_text, target_text

    return seqs.map(split_input_target)

dataset = create_sequences(data)

# 6. Preparar el dataset para el entrenamiento
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# 7. Definir el modelo
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim,
                                  batch_input_shape=[batch_size, None]),
        tf.keras.layers.GRU(rnn_units,
                            return_sequences=True,
                            stateful=True,
                            recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(vocab_size)
    ])
    return model

EMBEDDING_DIM = 256
RNN_UNITS = 1024

model = build_model(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    rnn_units=RNN_UNITS,
    batch_size=BATCH_SIZE)

# 8. Función de pérdida
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

# 9. Callbacks
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

reset_states_callback = ResetStatesCallback()

# 10. Entrenar el modelo
EPOCHS = 30

model.fit(dataset, epochs=EPOCHS, callbacks=[reset_states_callback])

# 11. Generar texto
def generate_text(model, start_string, num_generate=50):
    # Evaluar el modelo
    # Para generar, la forma del input debe ser (batch_size, sequence_length)
    input_eval = [tokenizer.word_index[s] for s in start_string.split()]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # La temperatura controla la aleatoriedad de la predicción
    # Experimenta con diferentes valores
    temperature = 1.0

    # Aquí el batch size == 1
    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        # Eliminar la dimensión del batch
        predictions = tf.squeeze(predictions, 0)

        # Usar una distribución categórica para predecir la palabra
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # Pasar la palabra predicha como la siguiente entrada al modelo
        # junto con el estado oculto anterior
        input_eval = tf.expand_dims([predicted_id], 0)

        text_generated.append(tokenizer.index_word[predicted_id])

    return (start_string + ' ' + ' '.join(text_generated))

# Ejemplo de uso
start_string = "En un lugar"
print(generate_text(model, start_string))

2025-04-23 19:41:16.296943: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-23 19:41:36.071863: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


Epoch 1/30
13/28 [============>.................] - ETA: 8:23 - loss: 8.1299



**Cambios principales:**

*   **Tokenización por palabras:** Se utiliza `tf.keras.preprocessing.text.Tokenizer` para tokenizar el texto en palabras.
*   **`VOCAB_SIZE`:** Se calcula el tamaño del vocabulario a partir del `word_index` del tokenizer.
*   **`sequences`:** El texto se convierte a una secuencia de índices de palabras.
*   **`generate_text`:** La función de generación de texto se ha modificado para generar palabras en lugar de caracteres.  Utiliza el `tokenizer.index_word` para convertir los índices predichos a palabras.

**Explicación de la función `generate_text`:**

1.  **Preparar la entrada:**
    *   Convierte la cadena de inicio (`start_string`) en una secuencia de índices de palabras utilizando el `tokenizer`.
    *   Expande las dimensiones de la secuencia para que tenga la forma `(1, sequence_length)`, que es la forma esperada por el modelo.
2.  **Inicializar la salida:**
    *   Crea una lista vacía (`text_generated`) para almacenar las palabras generadas.
3.  **Controlar la aleatoriedad:**
    *   Define un parámetro `temperature` para controlar la aleatoriedad de la generación. Valores más bajos hacen que el texto sea más predecible, mientras que valores más altos lo hacen más aleatorio.
4.  **Loop de generación:**
    *   Itera `num_generate` veces para generar la cantidad de palabras especificada.
    *   **Predicción:** Pasa la secuencia de entrada al modelo para obtener las predicciones.
    *   **Eliminar la dimensión del lote:** Elimina la dimensión del lote de las predicciones.
    *   **Ajustar la temperatura:** Divide las predicciones por la temperatura para controlar la aleatoriedad.
    *   **Muestreo:** Utiliza `tf.random.categorical` para muestrear un índice de palabra de la distribución de probabilidad predicha.
    *   **Convertir a palabra:** Convierte el índice predicho a una palabra utilizando `tokenizer.index_word`.
    *   **Actualizar la entrada:** Crea una nueva secuencia de entrada que contiene solo la palabra predicha.
    *   **Añadir a la salida:** Añade la palabra predicha a la lista de palabras generadas.
5.  **Devolver el texto generado:**
    *   Combina la cadena de inicio con las palabras generadas para crear el texto final.

Este código proporciona una base para la generación de texto palabra a palabra con Keras. Puedes experimentar con diferentes arquitecturas de modelo, hiperparámetros y técnicas de generación para mejorar el rendimiento.


### Pytorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import nltk

# 1. Cargar el texto
filepath = "./el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Tokenización por palabras
tokens = nltk.word_tokenize(el_quijote_text)

# Crear vocabulario
word_counts = {}
for token in tokens:
    if token not in word_counts:
        word_counts[token] = 0
    word_counts[token] += 1

# Filtrar palabras poco frecuentes (opcional)
min_frequency = 5
filtered_tokens = [token for token in tokens if word_counts[token] >= min_frequency]

vocab = sorted(set(filtered_tokens))
word_to_index = {word: index for index, word in enumerate(vocab)}
index_to_word = {index: word for index, word in enumerate(vocab)}

VOCAB_SIZE = len(vocab)

# Convertir texto a índices
text_as_int = np.array([word_to_index[word] for word in filtered_tokens])

# 3. Dataset y DataLoader
class QuijoteDataset(Dataset):
    def __init__(self, text_as_int, seq_length):
        self.text_as_int = text_as_int
        self.seq_length = seq_length
        self.examples_per_epoch = len(text_as_int) // (seq_length + 1)
        self.num_sequences = self.examples_per_epoch

        # Generar los datos
        self.input_seqs = []
        self.target_seqs = []
        for i in range(self.num_sequences):
            start_index = i * (seq_length + 1)
            end_index = start_index + seq_length + 1
            chunk = text_as_int[start_index:end_index]
            input_text = chunk[:-1]
            target_text = chunk[1:]
            self.input_seqs.append(input_text)
            self.target_seqs.append(target_text)

    def __len__(self):
        return len(self.input_seqs)

    def __getitem__(self, idx):
        return torch.tensor(self.input_seqs[idx], dtype=torch.long), torch.tensor(self.target_seqs[idx], dtype=torch.long)

BATCH_SIZE = 64
SEQ_LENGTH = 100
dataset = QuijoteDataset(text_as_int, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, drop_last=True)

# 4. Definir el modelo
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super(CharRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, rnn_units, batch_first=True)
        self.dense = nn.Linear(rnn_units, vocab_size)
        self.rnn_units = rnn_units

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded, hidden)
        output = self.dense(output)
        return output, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.rnn_units)

# 5. Hiperparámetros
EMBEDDING_DIM = 256
RNN_UNITS = 1024
LEARNING_RATE = 0.001
NUM_EPOCHS = 30

# 6. Instanciar el modelo
model = CharRNN(VOCAB_SIZE, EMBEDDING_DIM, RNN_UNITS)

# 7. Definir optimizador y función de pérdida
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# 8. Dispositivo (GPU si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 9. Loop de entrenamiento
for epoch in range(NUM_EPOCHS):
    # Inicializar el estado oculto
    hidden = model.init_hidden(BATCH_SIZE).to(device)

    for batch_idx, (input_data, target_data) in enumerate(dataloader):
        input_data = input_data.to(device)
        target_data = target_data.to(device)

        # Limpiar los gradientes
        optimizer.zero_grad()

        # Forward pass
        output, hidden = model(input_data, hidden)

        # Calcular la pérdida
        loss = criterion(output.view(-1, VOCAB_SIZE), target_data.view(-1))

        # Backward pass y optimización
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {loss.item():.4f}')

    # Detach hidden state (important for stateful RNNs)
    hidden = hidden.detach()

# 10. Generar texto
def generate_text(model, start_string, num_generate=50, temperature=1.0):
    model.eval()  # Poner el modelo en modo de evaluación
    model.to(device)

    # Tokenizar la cadena de inicio
    tokens = nltk.word_tokenize(start_string)
    input_indices = [word_to_index[word] for word in tokens if word in word_to_index]  # Ignorar palabras desconocidas

    # Convertir a tensor y mover al dispositivo
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)  # Agregar dimensión de batch

    # Inicializar el estado oculto
    hidden = model.init_hidden(1).to(device)  # Batch size = 1 para la generación

    # Generar texto
    generated_text = []
    with torch.no_grad():  # Desactivar el cálculo de gradientes
        for i in range(num_generate):
            # Forward pass
            output, hidden = model(input_tensor, hidden)

            # Ajustar la temperatura
            output = output / temperature

            # Muestrear de la distribución de probabilidad
            probabilities = torch.softmax(output[0, -1], dim=0)  # Obtener las probabilidades del último token
            predicted_index = torch.multinomial(probabilities, 1).item()

            # Convertir el índice a palabra
            predicted_word = index_to_word[predicted_index]
            generated_text.append(predicted_word)

            # Preparar la siguiente entrada
            input_tensor = torch.tensor([predicted_index], dtype=torch.long).unsqueeze(0).to(device)

    return start_string + ' ' + ' '.join(generated_text)

# Ejemplo de uso
start_string = "En un lugar"
print(generate_text(model, start_string))



**Cambios principales:**

*   **Tokenización por palabras:** Se utiliza `nltk.word_tokenize` para tokenizar el texto en palabras.
*   **`VOCAB_SIZE`:** Se calcula el tamaño del vocabulario a partir del `word_to_index`.
*   **`text_as_int`:** El texto se convierte a una secuencia de índices de palabras.
*   **`generate_text`:** La función de generación de texto se ha modificado para generar palabras en lugar de caracteres.  Utiliza el `index_to_word` para convertir los índices predichos a palabras.

**Explicación de la función `generate_text`:**

1.  **Preparar la entrada:**
    *   Tokeniza la cadena de inicio (`start_string`) en palabras.
    *   Convierte las palabras en índices usando `word_to_index`.
    *   Crea un tensor de entrada a partir de los índices y lo mueve al dispositivo.
2.  **Inicializar el estado oculto:**
    *   Inicializa el estado oculto del modelo.
3.  **Loop de generación:**
    *   Itera `num_generate` veces para generar la cantidad de palabras especificada.
    *   **Forward pass:** Pasa el tensor de entrada y el estado oculto al modelo para obtener la salida y el nuevo estado oculto.
    *   **Ajustar la temperatura:** Divide la salida por la temperatura para controlar la aleatoriedad.
    *   **Muestreo:** Utiliza `torch.multinomial` para muestrear un índice de palabra de la distribución de probabilidad predicha.
    *   **Convertir a palabra:** Convierte el índice predicho a una palabra utilizando `index_to_word`.
    *   **Preparar la siguiente entrada:** Crea un nuevo tensor de entrada que contiene solo el índice de la palabra predicha.
4.  **Devolver el texto generado:**
    *   Combina la cadena de inicio con las palabras generadas para crear el texto final.

**Puntos importantes:**

*   **`model.eval()`:** Es importante poner el modelo en modo de evaluación antes de generar texto. Esto desactiva el dropout y otras capas que se comportan de manera diferente durante el entrenamiento y la evaluación.
*   **`torch.no_grad()`:** El bloque `with torch.no_grad():` desactiva el cálculo de gradientes durante la generación de texto. Esto reduce el consumo de memoria y acelera el proceso.
*   **`hidden = hidden.detach()`:** Aunque no estamos entrenando durante la generación, es una buena práctica desconectar el estado oculto del grafo computacional para evitar problemas de memoria.

Este código proporciona una base para la generación de texto palabra a palabra con PyTorch. Puedes experimentar con diferentes arquitecturas de modelo, hiperparámetros y técnicas de generación para mejorar el rendimiento.

### Fast.ai

In [ ]:
from fastai.text.all import *
import numpy as np
import nltk

# 1. Cargar el texto
filepath = "./el_quijote.txt"
with open(filepath, 'r', encoding='utf-8') as f:
    el_quijote_text = f.read()

# 2. Tokenización por palabras
tokens = nltk.word_tokenize(el_quijote_text)

# Crear vocabulario
word_counts = {}
for token in tokens:
    if token not in word_counts:
        word_counts[token] = 0
    word_counts[token] += 1

# Filtrar palabras poco frecuentes (opcional)
min_frequency = 5
filtered_tokens = [token for token in tokens if word_counts[token] >= min_frequency]

vocab = sorted(set(filtered_tokens))
word_to_index = {word: index for index, word in enumerate(vocab)}
index_to_word = {index: word for index, word in enumerate(vocab)}

VOCAB_SIZE = len(vocab)

# 3. Preparar los datos para fastai
# Crear un DataLoaders
bs=64 #Tamaño del batch
sl=100 #Longitud de la secuencia

# Convertir el texto a índices
text_as_int = [word_to_index[word] for word in filtered_tokens]

# Función para crear el DataLoader
def get_dls(text, bs=64, seq_len=80):
    source = L(text).map(TensorText)
    splits = [o for o in range_of(source) if o % 10 == 0]
    tls = TfmdLists(source, vocab=Vocab(vocab), splits=splits, dl_type=LMDataLoader)
    return tls.dataloaders(bs=bs, seq_len=seq_len)

dls = get_dls(text_as_int, bs=bs, seq_len=sl)

# 4. Crear un modelo
learn = language_model_learner(
    dls, AWD_LSTM, drop_mult=0.3,
    metrics=[accuracy, Perplexity()],
    pretrained=False) #No usar modelo preentrenado

# 5. Encontrar una buena tasa de aprendizaje
learn.lr_find()

# 6. Entrenar el modelo
learn.fit_one_cycle(1, 1e-2) #Ajustar la tasa de aprendizaje

# 7. Generar texto
def generate_text(learn, start_string, num_words=50, temperature=1.0):
    prompt = start_string
    preds = learn.predict(prompt, n_words=num_words, temperature=temperature)
    return prompt + ' ' + preds

# Ejemplo de uso
start_string = "En un lugar"
print(generate_text(learn, start_string))



**Cambios principales:**

*   **Tokenización por palabras:** Se utiliza `nltk.word_tokenize` para tokenizar el texto en palabras.
*   **`VOCAB_SIZE`:** Se calcula el tamaño del vocabulario a partir del `word_to_index`.
*   **`text_as_int`:** El texto se convierte a una secuencia de índices de palabras.
*   **`get_dls`:** Se crea una función para generar el `DataLoader` a partir de la secuencia de índices de palabras.
*   **`generate_text`:** La función de generación de texto se ha modificado para generar palabras en lugar de caracteres.

**Explicación detallada:**

1.  **Importar librerías:** Importa las librerías necesarias de `fastai` y `nltk`.
2.  **Cargar el texto:** Carga el texto de El Quijote desde el archivo.
3.  **Tokenización por palabras:** Se utiliza `nltk.word_tokenize` para tokenizar el texto en palabras.
4.  **Crear vocabulario:** Crea el vocabulario de palabras únicas.
5.  **Preparar los datos para fastai:**
    *   **`text_as_int`:** Convierte el texto a una lista de índices de palabras.
    *   **`get_dls`:** Esta función crea un `DataLoaders` a partir de una lista de índices de palabras.
        *   `L(text).map(TensorText)`: Convierte la lista de índices en un `L` (una lista de `fastai`) y luego aplica la función `TensorText` a cada elemento para convertirlo en un tensor.
        *   `splits = [o for o in range_of(source) if o % 10 == 0]`: Crea una lista de índices para dividir los datos en entrenamiento y validación.
        *   `TfmdLists(source, vocab=Vocab(vocab), splits=splits, dl_type=LMDataLoader)`: Crea un `TfmdLists` (una lista transformada) que contiene los datos, el vocabulario y las divisiones.
        *   `tls.dataloaders(bs=bs, seq_len=seq_len)`: Crea el `DataLoaders` a partir del `TfmdLists`.
6.  **Crear un modelo:**
    *   **`language_model_learner`:** Esta función crea un modelo de lenguaje usando la arquitectura AWD\_LSTM.
7.  **Encontrar una buena tasa de aprendizaje:**
    *   **`learn.lr_find()`:** Esta función ayuda a encontrar una buena tasa de aprendizaje para el modelo.
8.  **Entrenar el modelo:**
    *   **`learn.fit_one_cycle`:** Esta función entrena el modelo usando la técnica "one cycle policy".
9.  **Generar texto:**
    *   **`generate_text`:** Esta función genera texto a partir de un prompt.
        *   `learn.predict(prompt, n_words=num_words, temperature=temperature)`: Utiliza la función `predict` de `fastai` para generar texto.

**Puntos clave:**

*   **`DataLoaders`:** `fastai` usa `DataLoaders` para manejar los datos. Es importante crear el `DataLoaders` correctamente para que el modelo pueda entrenar.
*   **`language_model_learner`:** Esta función simplifica la creación de modelos de lenguaje.
*   **`fit_one_cycle`:** Esta función entrena el modelo usando una política de tasa de aprendizaje que suele dar buenos resultados.
*   **`predict`:** Esta función genera texto a partir de un prompt.

Este código proporciona una forma sencilla de entrenar un modelo generativo de texto palabra a palabra con `fastai`. Puedes experimentar con diferentes hiperparámetros, arquitecturas de modelo y técnicas de generación para mejorar el rendimiento.